# MPSQCL Views Ablation - Continuation (MotionSense M=6, MobiAct, USC-HAD)
Continues the multi-view ablation study from where the previous session timed out.

**Completed in previous session**: UCI-HAR (all), SHAR (all), MotionSense (M=2-5)

**Remaining**: MotionSense M=6, MobiAct (all), USC-HAD (all)

Pre-trained encoder checkpoints are saved with `--save_checkpoints` for later LSTM fine-tuning.

In [ ]:
# Step 1: Install Dependencies & Verify GPU
!pip install pennylane scikit-learn matplotlib scipy -q

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}")
else:
    print("WARNING: No GPU! Select GPU T4 x2 under Notebook Options.")

In [ ]:
# Step 2: Clone Repo & Link Kaggle Input Datasets
import os, sys

if not os.path.exists("QMLHAR") and not os.path.exists("experiments"):
    !git clone https://github.com/Maximus-08/QMLHAR.git
    %cd QMLHAR
elif os.path.exists("QMLHAR"):
    %cd QMLHAR

if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)

# Symlink datasets from Kaggle input
target_dirs = ["UCI-HAR Dataset", "unimib_shar", "motion_sense", "MobiAct", "USC-HAD"]
target_files = {
    "uniMiB-SHAR.mat": "unimib_shar",
    "motionsense_preprocessed.npz": "motion_sense",
    "mobiact_preprocessed.npz": "MobiAct",
    "uschad_preprocessed.npz": "USC-HAD",
}

for root, dirs, files in os.walk("/kaggle/input"):
    for d in dirs:
        if d in target_dirs:
            src = os.path.join(root, d)
            dst = os.path.join("data", d)
            if not os.path.exists(dst):
                print(f"Symlinking {src} -> {dst}")
                os.symlink(src, dst)
    for f in files:
        if f in target_files:
            src_file = os.path.join(root, f)
            target_subdir = target_files[f]
            dst_dir = os.path.join("data", target_subdir)
            dst_file = os.path.join(dst_dir, f)
            if not os.path.exists(dst_file):
                os.makedirs(dst_dir, exist_ok=True)
                print(f"Symlinking {src_file} -> {dst_file}")
                os.symlink(src_file, dst_file)

print(f"data/ contents: {os.listdir('data')}")
print(f"Working Directory: {os.getcwd()}")

In [ ]:
# Step 3A: Complete MotionSense M=6 (only remaining view)
!python -u experiments/run_mpsqcl_views_ablation.py \
    --dataset motionsense \
    --subset_fraction 1.0 \
    --epochs_pretrain 75 \
    --epochs_finetune 50 \
    --batch_size 64 \
    --views 6 \
    --save_checkpoints \
    --output_file results/mpsqcl_views_ablation_results_motionsense_m6.md

In [ ]:
# Step 3B: MobiAct Full Sweep (M=2,3,4,5,6)
!python -u experiments/run_mpsqcl_views_ablation.py \
    --dataset mobiact \
    --subset_fraction 1.0 \
    --epochs_pretrain 75 \
    --epochs_finetune 50 \
    --batch_size 64 \
    --views 2 3 4 5 6 \
    --save_checkpoints \
    --output_file results/mpsqcl_views_ablation_results_mobiact.md

In [ ]:
# Step 3C: USC-HAD Full Sweep (M=2,3,4,5,6)
!python -u experiments/run_mpsqcl_views_ablation.py \
    --dataset uschad \
    --subset_fraction 1.0 \
    --epochs_pretrain 75 \
    --epochs_finetune 50 \
    --batch_size 64 \
    --views 2 3 4 5 6 \
    --save_checkpoints \
    --output_file results/mpsqcl_views_ablation_results_uschad.md

In [ ]:
# Step 4: Display All Results & Package Output
import shutil

result_files = [
    'results/mpsqcl_views_ablation_results_motionsense_m6.md',
    'results/mpsqcl_views_ablation_results_mobiact.md',
    'results/mpsqcl_views_ablation_results_uschad.md',
]

for rf in result_files:
    if os.path.exists(rf):
        print(f"\n=== {rf} ===")
        with open(rf) as f:
            print(f.read())
    else:
        print(f"Not found: {rf}")

# List saved checkpoints
print("\n=== Saved Checkpoints ===")
for f in os.listdir('results'):
    if f.endswith('.pt'):
        size_mb = os.path.getsize(os.path.join('results', f)) / (1024*1024)
        print(f"  {f} ({size_mb:.1f} MB)")

# Package everything
shutil.make_archive('/kaggle/working/views_ablation_continuation_results', 'zip', 'results')
print("\nResults packaged to /kaggle/working/views_ablation_continuation_results.zip")